# Week 5 - Spark Data Cleaning Assignment
**Objective:** Understand Spark fundamentals and use it to clean, transform, and analyze data using DataFrames.

This notebook covers:
- Loading data into a Spark DataFrame
- Data cleaning (duplicates, nulls, inconsistent formats)
- Filtering, transformation, and aggregation
- Answers to Q1-Q15 from the assignment


## Step 1-2: Setup Spark Session and Load Data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Week5-DataCleaning").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("../data/dataset.csv", header=True, inferSchema=True)
print("Total rows:", df.count())
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/16 19:35:33 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/16 19:35:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/07/16 19:35:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Total rows: 620
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [2]:
# First look at the data
df.show(5)

+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|         email|username|  price|store_id|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|  U1363|      2024-06-16|  West|       Furniture|     365.23|Bangalore| 25|     Premium| Pending|         16/06/2024|u1363@mail.com|user_363|2914.57|    ST10|
|  U1133|      2024-01-19| South|            Toys|    2306.74|Bangalore| 41|     Premium|Inactive|2024-01-19 00:00:00|u1133@mail.com|user_133| 2819.9|     ST3|
|  U1470|      2024-02-11|  West|         Grocery|     1869.7|  Chennai| 45|       Basic|  Active|2024-02-11T00:00:00|u1470@mail.com|user_470|   NULL|     ST4|
|  U1054|      2024-04-16|  East|       

## Q1: Limitations of MapReduce vs Spark
**Answer (theory):**
- MapReduce writes intermediate results to disk after every map/reduce step, causing high I/O overhead, especially for iterative algorithms.
- Spark keeps data in-memory (RDD/DataFrame caching) across stages, avoiding repeated disk reads/writes.
- Spark's DAG execution engine optimizes the whole pipeline instead of running rigid map-then-reduce stages.
- Spark supports interactive queries, streaming, ML, and graph processing in one unified engine, while MapReduce is batch-only.
- Because of in-memory computation, Spark can be 10-100x faster than MapReduce for iterative workloads like ML training.

## Q2: How Spark's In-Memory Computing speeds up iterative ML
**Answer (theory):**
- In disk-based systems (like classic MapReduce), each iteration of an algorithm (e.g., gradient descent, k-means) reads data from disk, processes it, and writes results back to disk before the next iteration starts.
- Spark keeps the DataFrame/RDD cached in RAM (`.cache()` / `.persist()`) after the first pass, so subsequent iterations reuse the in-memory copy instead of re-reading from disk.
- This drastically cuts I/O latency, which is the main bottleneck in iterative ML algorithms that need to pass over the same dataset many times.

In [3]:
# Example: caching a DataFrame for repeated iterative access
df_cached = df.cache()
df_cached.count()  # triggers the cache to materialize

620

## Q3: Remove duplicate rows based on `user_id` and `transaction_date`

In [4]:
df_q3 = df.dropDuplicates(["user_id", "transaction_date"])
print("Rows before:", df.count(), "-> Rows after dropDuplicates:", df_q3.count())
df_q3.show(5)

Rows before: 620 -> Rows after dropDuplicates: 600


+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|         email|username|  price|store_id|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|  U1001|      2024-06-12|  West|     Electronics|     3733.6|   Mumbai| 19|     Premium|  Active|         2024-06-12|          NULL|  user_1|1825.96|     ST9|
|  U1002|      2024-02-20| South|        Clothing|    2301.12|    Delhi| 42|     Premium|Inactive|         2024-02-20|u1002@mail.com|  user_2|1042.95|     ST2|
|  U1003|      2024-04-07|  West|         Grocery|    4252.72|Bangalore| 40|     Premium|    NULL|2024-04-07 00:00:00|u1003@mail.com|  user_3|2496.74|    ST10|
|  U1004|      2024-04-02|  East|     El

## Q4: Filter `region == 'West'`, group by `product_category`, average `sale_amount`

In [5]:
df_q4 = df.filter(F.col("region") == "West") \
    .groupBy("product_category") \
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))
df_q4.show()

+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|         Grocery|2620.1722857142863|
|     Electronics|2514.7845714285713|
|        Clothing|        2510.72125|
|       Furniture| 2469.603461538462|
|            Toys| 2281.297931034482|
+----------------+------------------+



## Q5: `.na.drop()` vs `.na.fill()`
**Answer (theory):**
- `.na.drop()` removes rows that contain null values (in any or specified columns). Useful when incomplete rows are unusable and dropping them doesn't bias the dataset significantly.
- `.na.fill()` replaces nulls with a specified default value, preserving the row. Useful when the row still has valuable information in other columns.

Below: filling nulls in the `status` column with `'Unknown'`.

In [6]:
df_q5 = df.na.fill({"status": "Unknown"})
print("Nulls in status before:", df.filter(F.col("status").isNull()).count())
print("Nulls in status after fill:", df_q5.filter(F.col("status").isNull()).count())
df_q5.select("user_id", "status").show(5)

Nulls in status before: 175


Nulls in status after fill: 0


+-------+--------+
|user_id|  status|
+-------+--------+
|  U1363| Pending|
|  U1133|Inactive|
|  U1470|  Active|
|  U1054| Pending|
|  U1267| Pending|
+-------+--------+
only showing top 5 rows


## Q6: Total count of records per `city`, keep only cities with count > 100

In [7]:
df_q6 = df.groupBy("city").count().filter(F.col("count") > 100)
df_q6.show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|  102|
|   Mumbai|  152|
|    Delhi|  114|
+---------+-----+



## Q7: Immutability of Spark DataFrames and data cleaning
**Answer (theory):**
- Spark DataFrames are immutable - once created, they cannot be changed in place.
- Every "cleaning" operation (dropping a column, renaming, filtering) actually returns a new DataFrame; the original is untouched.
- This means we always chain transformations and reassign to a new variable, e.g. `df2 = df.drop("col")`, rather than modifying `df` directly.
- Immutability makes Spark pipelines safer for parallel/distributed execution (no risk of concurrent writes to the same data) and makes transformations easy to reason about and debug, since each step produces a traceable new DataFrame.

## Q8: Filter rows where `age` is between 18-30 (inclusive) and `subscription == 'Premium'`

In [8]:
df_q8 = df.filter((F.col("age").between(18, 30)) & (F.col("subscription") == "Premium"))
print("Matching rows:", df_q8.count())
df_q8.show(5)

Matching rows: 118


+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|  status|      raw_timestamp|         email|username|  price|store_id|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------+-------------------+--------------+--------+-------+--------+
|  U1363|      2024-06-16|  West|       Furniture|     365.23|Bangalore| 25|     Premium| Pending|         16/06/2024|u1363@mail.com|user_363|2914.57|    ST10|
|  U1452|      2024-02-14|  West|            Toys|    3575.75|   Mumbai| 25|     Premium|  Active|         2024-02-14|u1452@mail.com|user_452|2123.39|     ST4|
|  U1492|      2024-02-24| North|         Grocery|      332.9|Ahmedabad| 27|     Premium|  Active|2024-02-24 00:00:00|u1492@mail.com|user_492|1616.64|     ST7|
|  U1403|      2024-04-29|  East|     El

## Q9: Why handle nulls before aggregations like `sum()` / `avg()`
**Answer (theory):**
- Functions like `sum()` and `avg()` in Spark skip nulls by default, but if nulls are widespread, this silently changes the denominator/count behind the scenes and can produce misleading results.
- Null values can also propagate into derived columns (e.g., `col_a + col_b` is null if either is null), corrupting downstream calculations.
- Cleaning nulls upfront (via `.na.fill()` or `.na.drop()`) ensures aggregation results are consistent, predictable, and correctly reflect the intended business logic - rather than silently dropping rows or including 0s where they shouldn't be.

## Q10: Cast `raw_timestamp` to `TimestampType` and rename to `event_time`
Note: our `raw_timestamp` column has inconsistent formats (some `yyyy-MM-dd HH:mm:ss`, some `dd/MM/yyyy`, some ISO format, some plain date).
We use `try_to_timestamp` with multiple format attempts + `coalesce` so mismatched formats don't crash the job - they fall through to the next pattern.

In [9]:
df_q10 = df.withColumn(
    "event_time",
    F.coalesce(
        F.expr("try_to_timestamp(raw_timestamp, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(raw_timestamp, \"yyyy-MM-dd'T'HH:mm:ss\")"),
        F.expr("try_to_timestamp(raw_timestamp, 'dd/MM/yyyy')"),
        F.expr("try_to_timestamp(raw_timestamp, 'yyyy-MM-dd')")
    )
).drop("raw_timestamp")

df_q10.select("event_time").show(5, truncate=False)
print("Unparseable timestamps (still null):", df_q10.filter(F.col("event_time").isNull()).count())

+-------------------+
|event_time         |
+-------------------+
|2024-06-16 00:00:00|
|2024-01-19 00:00:00|
|2024-02-11 00:00:00|
|2024-04-16 00:00:00|
|2024-06-20 00:00:00|
+-------------------+
only showing top 5 rows


Unparseable timestamps (still null): 0


## Q11: The "Shuffle" process during grouping
**Answer (theory):**
- When you run `groupBy()` followed by an aggregation, Spark needs to bring together all rows with the same key so they can be aggregated - but those rows are usually scattered across different partitions/executors.
- Shuffle is the process of physically redistributing data across the cluster (writing to disk, transferring over the network, reading back) so rows with the same key end up on the same partition.
- It's called a wide transformation because each output partition can depend on data from multiple input partitions (unlike a narrow transformation like `filter()`, where each output partition depends on only one input partition).
- Shuffles are expensive (network I/O + disk I/O), so minimizing unnecessary `groupBy`/`join`/`repartition` calls is a key Spark performance practice.

## Q12: Remove rows where `email` is null OR `username` is an empty string

In [10]:
df_q12 = df.filter(~(F.col("email").isNull() | (F.col("username") == "")))
print("Rows before:", df.count(), "-> Rows after cleaning:", df_q12.count())
df_q12.select("user_id", "email", "username").show(5)

Rows before: 620 -> Rows after cleaning: 532


+-------+--------------+--------+
|user_id|         email|username|
+-------+--------------+--------+
|  U1363|u1363@mail.com|user_363|
|  U1133|u1133@mail.com|user_133|
|  U1470|u1470@mail.com|user_470|
|  U1054|u1054@mail.com| user_54|
|  U1267|u1267@mail.com|user_267|
+-------+--------------+--------+
only showing top 5 rows


## Q13: Use `.agg()` to calculate min, max, and mean of the `price` column

In [11]:
df_q13 = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("mean_price")
)
df_q13.show()

+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|    62.12|  2992.53|1477.0969534050178|
+---------+---------+------------------+



## Q14: Risk of `inferSchema=True` with messy/inconsistent date formats
**Answer (theory):**
- `inferSchema=True` samples the data and guesses column types. If a date column has mixed formats (e.g., `2024-01-15`, `15/01/2024`, `2024-01-15T10:30:00`), Spark often can't confidently infer a single `DateType`/`TimestampType` and falls back to reading the column as a plain string.
- Even if it does infer a date type, rows that don't match the inferred format can silently become null instead of raising an error - leading to silent data loss that's easy to miss.
- Schema inference also requires an extra pass over the data (or a sample of it) before reading, adding overhead - especially costly on large datasets.
- Best practice: define an explicit schema (`StructType`) when formats are known to be inconsistent, and parse date/timestamp columns manually (like we did in Q10 with `try_to_timestamp` + `coalesce`) instead of relying on automatic inference.

## Q15: Final processing pipeline
1. Filter out duplicates (on `user_id` + `transaction_date`)
2. Fill null `price` values with 0
3. Group by `store_id` to calculate total revenue

In [12]:
df_final = df.dropDuplicates(["user_id", "transaction_date"]) \
    .na.fill({"price": 0}) \
    .groupBy("store_id") \
    .agg(F.sum("price").alias("total_revenue")) \
    .orderBy("store_id")

df_final.show()

+--------+-----------------+
|store_id|    total_revenue|
+--------+-----------------+
|     ST1|         74620.59|
|    ST10|84338.09000000001|
|     ST2|         82743.53|
|     ST3|99221.31000000003|
|     ST4|68508.20999999999|
|     ST5|76661.84999999999|
|     ST6|84674.32999999999|
|     ST7|62217.47000000001|
|     ST8|92759.90000000001|
|     ST9|         79141.23|
+--------+-----------------+



In [13]:
# Save the final pipeline output
df_final.toPandas().to_csv("../output/results.csv", index=False)
print("Saved to ../output/results.csv")

/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved to ../output/results.csv


## Insights / Observations
- Original dataset had 620 rows; after de-duplicating on `user_id` + `transaction_date`, 600 unique transactions remained (20 duplicate rows removed).
- Only Mumbai, Delhi, and Bangalore crossed the 100-transaction mark among all cities.
- Filtering age 18-30 + Premium subscription narrowed the dataset to a meaningful high-value customer segment for targeted analysis.
- `raw_timestamp` had 4 different formats - handling this with `try_to_timestamp` + `coalesce` avoided job failures instead of relying on default `inferSchema` behavior (directly relevant to Q14).
- The final revenue pipeline shows `store_id`-wise totals after cleaning, ready for further reporting/BI use.

In [14]:
spark.stop()